# Task 8 – Collaborative Filtering Recommendation Model

## 1. Introduction

The objective of this task is to build a User-Based
Collaborative Filtering model that recommends TED Talks
based on user preferences and interaction history.

The interaction dataset prepared in Task 7 is used.
The dataset contains simulated user interactions.

In [26]:

# Step 1: Import required libraries
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display


In [27]:
# Step 2: Load the interaction dataset

interactions = pd.read_csv(
    "../data/interaction/user_interactions.csv"
)

print("Dataset Shape:", interactions.shape)

display(interactions.head())


Dataset Shape: (32, 6)


,user_id,talk_id,rating,liked,watched,saved
0,1,995,5,1,1,1
1,1,1307,4,1,1,0
2,1,1879,2,0,1,0
3,1,41223,5,1,1,1
4,2,995,5,1,1,0


In [28]:
# Step 3: Inspect the dataset

print("Dataset Information:")
interactions.info()

print("\nMissing Values:")
print(interactions.isnull().sum())

print("\nUnique Users:",
      interactions["user_id"].nunique())

print("Unique TED Talks:",
      interactions["talk_id"].nunique())

print("Total Interactions:",
      len(interactions))


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   user_id  32 non-null     int64
 1   talk_id  32 non-null     int64
 2   rating   32 non-null     int64
 3   liked    32 non-null     int64
 4   watched  32 non-null     int64
 5   saved    32 non-null     int64
dtypes: int64(6)
memory usage: 1.6 KB

Missing Values:
user_id    0
talk_id    0
rating     0
liked      0
watched    0
saved      0
dtype: int64

Unique Users: 8
Unique TED Talks: 9
Total Interactions: 32


### Activity 2 – Create the User-Item Interaction Matrix
Step 1: Understand the matrix
A User-Item Matrix represents the ratings given by users to TED Talks.
- Rows = Users.
- Columns = TED Talks.
- Values = Ratings.
- Empty cells = No rating.

In [29]:
# Step 2: Generate the User-Item Matrix

user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating"
)

print("User-Item Interaction Matrix:")

display(user_item_matrix)


User-Item Interaction Matrix:


talk_id,995,1075,1307,1776,1879,2338,2441,32269,41223
user_id,,,,,,,,,
1,5.0,NaN,4.0,NaN,2.0,NaN,NaN,NaN,5.0
2,5.0,NaN,4.0,NaN,2.0,NaN,NaN,5.0,NaN
3,NaN,NaN,NaN,NaN,5.0,NaN,5.0,4.0,2.0
4,2.0,NaN,NaN,NaN,5.0,NaN,5.0,4.0,NaN
5,4.0,NaN,5.0,4.0,NaN,NaN,NaN,NaN,3.0
6,NaN,NaN,2.0,5.0,5.0,NaN,4.0,NaN,NaN
7,5.0,5.0,NaN,2.0,NaN,NaN,NaN,NaN,4.0
8,NaN,3.0,NaN,NaN,4.0,5.0,NaN,5.0,NaN


In [30]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating"
)

In [31]:
# Step 3: Check missing values

print("Missing Values in User-Item Matrix:")

print(user_item_matrix.isnull().sum().sum())

print("\nMatrix Shape:")
print(user_item_matrix.shape)


Missing Values in User-Item Matrix:
40

Matrix Shape:
(8, 9)


In [32]:
# Step 4: Fill missing values with zero

user_item_filled = user_item_matrix.fillna(0)

print("User-Item Matrix after Filling Missing Values:")

display(user_item_filled)


User-Item Matrix after Filling Missing Values:


talk_id,995,1075,1307,1776,1879,2338,2441,32269,41223
user_id,,,,,,,,,
1,5.0,0.0,4.0,0.0,2.0,0.0,0.0,0.0,5.0
2,5.0,0.0,4.0,0.0,2.0,0.0,0.0,5.0,0.0
3,0.0,0.0,0.0,0.0,5.0,0.0,5.0,4.0,2.0
4,2.0,0.0,0.0,0.0,5.0,0.0,5.0,4.0,0.0
5,4.0,0.0,5.0,4.0,0.0,0.0,0.0,0.0,3.0
6,0.0,0.0,2.0,5.0,5.0,0.0,4.0,0.0,0.0
7,5.0,5.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0
8,0.0,3.0,0.0,0.0,4.0,5.0,0.0,5.0,0.0


### Activity 3 – Calculate User Similarity
Now we will calculate how similar users are based on their rating patterns.

Step 1: Understand Cosine Similarity
Cosine Similarity measures how similar two users' rating patterns are.

Its values generally range from:
Score	Meaning
1.0	Very similar rating patterns
0.8	Highly similar
0.5	Moderately similar
0.0	No positive similarity
-1.0	Opposite directions (possible with other rating representations)

Our ratings are positive and missing values are zero, so the similarities in this implementation will be non-negative.

In [33]:
# Step 2: Calculate cosine similarity

similarity_array = cosine_similarity(user_item_filled)

user_similarity = pd.DataFrame(
    similarity_array,
    index=user_item_filled.index,
    columns=user_item_filled.index
)

print("User Similarity Matrix:")

display(user_similarity.round(2))


User Similarity Matrix:


user_id,1,2,3,4,5,6,7,8
user_id,,,,,,,,
1,1.00,0.64,0.29,0.29,0.81,0.26,0.64,0.11
2,0.64,1.00,0.43,0.57,0.59,0.26,0.36,0.46
3,0.29,0.43,1.00,0.94,0.09,0.64,0.11,0.55
4,0.29,0.57,0.94,1.00,0.12,0.64,0.14,0.55
5,0.81,0.59,0.09,0.12,1.00,0.44,0.59,0.00
6,0.26,0.26,0.64,0.64,0.44,1.00,0.14,0.28
7,0.64,0.36,0.11,0.14,0.59,0.14,1.00,0.21
8,0.11,0.46,0.55,0.55,0.00,0.28,0.21,1.00


Step 3: Understand the output
For example, imagine this similarity matrix:
User	 U1      U2	     U3
U1	    1.00	0.95	0.32
U2	    0.95	1.00	0.28
U3	    0.32	0.28	1.00

Interpretation:
- U1 and U2 have similarity 0.95.
- U1 and U3 have similarity 0.32.
- U1 is more similar to U2 than U3.
- Every user has similarity 1 with themselves.
The diagonal values are always 1 because each user is perfectly similar to themselves.

Why do we calculate similarity?

Because we want to find users who have similar interests and use their ratings to recommend new TED Talks.

### Activity 4 – Identify Similar Users

In [34]:
print("Available User IDs:")
print(user_similarity.index.tolist())

Available User IDs:
[1, 2, 3, 4, 5, 6, 7, 8]


In [35]:
# # Step 1: Select a user
selected_user = 1

similar_users = user_similarity[selected_user].drop(
    selected_user
).sort_values(ascending=False)

print("Users Similar to:", selected_user)

display(similar_users)

Users Similar to: 1


user_id
5    0.809174
7    0.642857
2    0.642857
4    0.285714
3    0.285714
6    0.257143
8    0.110410
Name: 1, dtype: float64

### Activity 5 – Develop the Recommendation Function
Step 1: Understand how recommendations are generated
Suppose:
- User U1 and U2 have similar preferences.
- U2 has rated a particular TED Talk 5.
- U1 has not rated that talk.


In [36]:
# Step 2: Write the recommendation function

def recommend_for_user(user_id, top_n=5):

    # Check whether the user exists
    if user_id not in user_item_matrix.index:
        raise ValueError(f"Invalid User ID: {user_id}")

    # Find similar users
    similar_users = user_similarity[user_id].drop(
        user_id
    ).sort_values(ascending=False)

    # Select talks not rated by the selected user
    unrated_talks = user_item_matrix.loc[
        user_id
    ][user_item_matrix.loc[user_id].isna()].index

    recommendations = []

    # Calculate score for each unrated talk
    for talk_id in unrated_talks:

        # Ratings given by other users
        ratings = user_item_matrix[talk_id]

        # Keep users who rated this talk
        common_users = ratings.dropna().index

        # Get similarity scores of those users
        common_users = [
            user for user in common_users
            if user in similar_users.index
            and similar_users[user] > 0
        ]

        if not common_users:
            continue

        scores = similar_users.loc[common_users]
        user_ratings = ratings.loc[common_users]

        # Similarity-weighted average
        predicted_rating = np.dot(
            scores, user_ratings
        ) / scores.sum()

        recommendations.append({
            "talk_id": talk_id,
            "predicted_rating": predicted_rating
        })

    # Convert recommendations into DataFrame
    result = pd.DataFrame(recommendations)

    if result.empty:
        return result

    # Sort by predicted rating
    result = result.sort_values(
        by="predicted_rating",
        ascending=False
    )

    return result.head(top_n).reset_index(drop=True)


In [37]:
# Step 3: Understand every important section
def recommend_for_user(user_id, top_n=5):

    # 1. Validate the user
    if user_id not in user_item_matrix.index:
        raise ValueError(f"Invalid User ID: {user_id}")

    # 2. Find similar users
    similar_users = user_similarity[user_id].drop(
        user_id
    ).sort_values(ascending=False)

    # 3. Find talks not rated by the selected user
    unrated_talks = user_item_matrix.loc[
        user_id
    ][user_item_matrix.loc[user_id].isna()].index

    recommendations = []

    # 4. Calculate scores for unrated talks
    for talk_id in unrated_talks:

        ratings = user_item_matrix[talk_id]

        common_users = ratings.dropna().index

        common_users = [
            user for user in common_users
            if user in similar_users.index
            and similar_users[user] > 0
        ]

        if not common_users:
            continue

        scores = similar_users.loc[common_users].values
        user_ratings = ratings.loc[common_users].values

        predicted_rating = np.dot(
            scores, user_ratings
        ) / scores.sum()

        recommendations.append({
            "talk_id": talk_id,
            "predicted_rating": predicted_rating
        })

    # 5. Create and sort results
    result = pd.DataFrame(
        recommendations,
        columns=["talk_id", "predicted_rating"]
    )

    if result.empty:
        return result

    result = result.sort_values(
        by="predicted_rating",
        ascending=False
    )

    return result.head(top_n).reset_index(drop=True)

### Activity 6 – Test the Recommendation Model

In [38]:
# Step 1: Test three users

test_users = user_item_matrix.index[:3]

for user in test_users:

    print("\nRecommendations for User:", user)

    recommendations = recommend_for_user(
        user_id=user,
        top_n=5
    )

    display(recommendations)



Recommendations for User: 1


,talk_id,predicted_rating
0,2338,5.000000
1,1075,4.706849
2,2441,4.689655
3,32269,4.568634
4,1776,3.398205



Recommendations for User: 2


,talk_id,predicted_rating
0,2338,5.000000
1,2441,4.795455
2,1075,3.879028
3,1776,3.619927
4,41223,3.602007



Recommendations for User: 3


,talk_id,predicted_rating
0,2338,5.000000
1,1776,4.490038
2,995,3.431553
3,1075,3.343026
4,1307,3.171560


In [39]:
# Step 2: Test whether already-rated talks are excluded

for user in test_users:

    recommendations = recommend_for_user(user, 5)

    already_rated = set(
        user_item_matrix.loc[user].dropna().index
    )

    recommended = set(recommendations["talk_id"])

    print("User:", user)

    print(
        "Already Rated Talks Excluded:",
        recommended.isdisjoint(already_rated)
    )


User: 1
Already Rated Talks Excluded: True
User: 2
Already Rated Talks Excluded: True
User: 3
Already Rated Talks Excluded: True


In [40]:
# Step 3: Test invalid User ID

try:
    recommend_for_user("U100", top_n=5)

except ValueError as error:
    print(error)


Invalid User ID: U100


In [41]:
# Step 4: Verify recommendation sorting

for user in test_users:

    recommendations = recommend_for_user(user, 5)

    if not recommendations.empty:

        scores = recommendations["predicted_rating"]

        print(
            user,
            "Recommendations Sorted:",
            scores.is_monotonic_decreasing
        )


1 Recommendations Sorted: True
2 Recommendations Sorted: True
3 Recommendations Sorted: True


In [42]:
# Step 5: Compare recommendations for different users

for user in test_users:

    recommendations = recommend_for_user(user, 5)

    print(f"\nUser: {user}")

    if not recommendations.empty:
        print(
            recommendations["talk_id"].tolist()
        )
    else:
        print("No recommendations available.")



User: 1
[2338, 1075, 2441, 32269, 1776]

User: 2
[2338, 2441, 1075, 1776, 41223]

User: 3
[2338, 1776, 995, 1075, 1307]


### Activity 7 – Display TED Talk Titles

In [43]:
# Step 1: Load your cleaned TED Talks dataset

talks = pd.read_csv(
    "../data/processed/ted_talks_cleaned.csv"
)

print("TED Talks Dataset:")
print(talks.shape)

print("\nAvailable Columns:")
print(talks.columns.tolist())

display(talks.head())


TED Talks Dataset:
(4005, 24)

Available Columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'description_clean', 'transcript_clean', 'topics_clean', 'combined_text']


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,topics,related_talks,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,with the same humor and humanity he exuded in ...,thank you so much chris and it's truly a great...,"['alternative energy', 'cars', 'climate change...",averting the climate crisis alternative energy...
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you've ever seen,you've never seen data presented like this wit...,about 10 years ago i took on the task to teach...,"['africa', 'asia', 'google', 'demo', 'economic...",the best stats you've ever seen africa asia go...
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...,"['computers', 'entertainment', 'interface desi...",simplicity sells computers entertainment inter...
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,in an emotionally charged talk macarthur-winni...,if you're here today and i'm very happy that y...,"['macarthur grant', 'activism', 'business', 'c...",greening the ghetto macarthur grant activism b...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity,sir ken robinson makes an entertaining and pro...,good morning how are you audience good it's be...,"['children', 'creativity', 'culture', 'dance',...",do schools kill creativity children creativity...


In [44]:
# Step 2: Check the ID column

if "talk_id" in talks.columns:
    id_column = "talk_id"

elif "id" in talks.columns:
    id_column = "id"

else:
    raise ValueError(
        "TED Talk ID column is missing. "
        "Preserve the original TED Talk IDs "
        "in the cleaned dataset before merging."
    )

if "title" not in talks.columns:
    raise ValueError("Title column is missing.")

talk_lookup = talks[
    [id_column, "title"]
].copy()

talk_lookup = talk_lookup.rename(
    columns={id_column: "talk_id"}
)

talk_lookup["talk_id"] = (
    talk_lookup["talk_id"].astype(str)
)

print("TED Talk ID and Title Mapping:")
display(talk_lookup.head())


TED Talk ID and Title Mapping:


,talk_id,title
0,1,Averting the climate crisis
1,92,The best stats you've ever seen
2,7,Simplicity sells
3,53,Greening the ghetto
4,66,Do schools kill creativity?


In [45]:
# Step 3: Combine recommendations with TED Talk titles

user_id = test_users[0]

recommendations = recommend_for_user(
    user_id=user_id,
    top_n=5
)

recommendations["talk_id"] = (
    recommendations["talk_id"].astype(str)
)

final_recommendations = recommendations.merge(
    talk_lookup,
    on="talk_id",
    how="left"
)

final_recommendations = final_recommendations[
    ["talk_id", "title", "predicted_rating"]
]

print("Final Recommendations for:", user_id)

display(final_recommendations)


Final Recommendations for: 1


,talk_id,title,predicted_rating
0,2338,How my mind came back to life — and no one knew,5.000000
1,1075,The linguistic genius of babies,4.706849
2,2441,The case for optimism on climate change,4.689655
3,32269,The uncomplicated truth about women's sexuality,4.568634
4,1776,Anatomy of a New Yorker cartoon,3.398205


In [46]:
# Step 4: Display recommendations for all three users with titles

for user in test_users:

    print(f"\nRecommendations for {user}")

    user_recommendations = recommend_for_user(
        user, top_n=5
    )

    user_recommendations["talk_id"] = (
        user_recommendations["talk_id"].astype(str)
    )

    result = user_recommendations.merge(
        talk_lookup,
        on="talk_id",
        how="left"
    )

    result = result[
        ["talk_id", "title", "predicted_rating"]
    ]

    display(result)



Recommendations for 1


,talk_id,title,predicted_rating
0,2338,How my mind came back to life — and no one knew,5.000000
1,1075,The linguistic genius of babies,4.706849
2,2441,The case for optimism on climate change,4.689655
3,32269,The uncomplicated truth about women's sexuality,4.568634
4,1776,Anatomy of a New Yorker cartoon,3.398205



Recommendations for 2


,talk_id,title,predicted_rating
0,2338,How my mind came back to life — and no one knew,5.000000
1,2441,The case for optimism on climate change,4.795455
2,1075,The linguistic genius of babies,3.879028
3,1776,Anatomy of a New Yorker cartoon,3.619927
4,41223,The mysterious science of pain,3.602007



Recommendations for 3


,talk_id,title,predicted_rating
0,2338,How my mind came back to life — and no one knew,5.000000
1,1776,Anatomy of a New Yorker cartoon,4.490038
2,995,Glorious visions in animation and performance,3.431553
3,1075,The linguistic genius of babies,3.343026
4,1307,Medicine for the 99 percent,3.171560


### Activity 8 – Analyze and Document Results

## Observations and Results

1. The user interaction dataset contains simulated
   interactions from 8 users and 9 TED Talks.

2. A User-Item Interaction Matrix was generated
   using user IDs, TED Talk IDs, and ratings.

3. Missing ratings were filled with zero for
   calculating cosine similarity.

4. Cosine similarity was used to identify users
   with similar rating patterns.

5. Users with higher similarity scores were
   considered more relevant for recommendations.

6. The recommendation function identifies TED
   Talks that the selected user has not rated.

7. A similarity-weighted average was used to
   calculate predicted ratings.

8. Recommendations were sorted in descending
   order of predicted ratings.

9. Three different users were tested to check
   the recommendation results.

10. The recommended TED Talk IDs were connected
    with the original dataset to display titles.

11. The model demonstrates how user preferences
    can be used to generate personalized
    recommendations.

### Limitations

1. The dataset contains simulated interactions
   rather than real user activity.

2. Only 8 users and a small number of TED Talks
   are available.

3. Filling missing ratings with zero may affect
   cosine similarity because missing interactions
   are not actual zero ratings.

4. New users without interaction history cannot
   receive personalized recommendations.

5. New TED Talks without ratings cannot be
   recommended through collaborative filtering.

6. User-Based Collaborative Filtering can become
   computationally expensive with large datasets.

7. The model uses ratings only for calculating
   recommendation scores. Like, watch, and save
   information are not included in the scoring.

## 9. Questions and Answers